# Step 1: Check the EDR export

This notebook belongs to the Geo-OBS manual, Session 6. Run it **directly after exporting the SEG-Y files from EDR**, before `2_fix_header_and_xyz.ipynb`. It only reads the first and the last trace header of every file, so it takes a few seconds.

**Why.** In 2026 one export started at 16:00:59 instead of 16:00:00. One minute of data was missing, nothing complained, and later all data of that station were 59 s too early. Another export used the clock skew of a different Trilobit, so its clock drift correction was wrong. Both mistakes are invisible in the data and easy to catch here.

**What counts as a problem** (per station, per chunk folder `1stChunk`, `2ndChunk`, ..., per component):

1. **Gap inside a file:** the last trace must start exactly `(number of traces - 1) x 60 s` after the first one.
2. **Gap between chunk folders:** each chunk must start at the latest where the previous one ends. An overlap is fine.
3. **Components differ:** `g1`, `g2`, `g3`, `h1` must have the same start, end and number of traces.
4. **Wrong `Trilobit_RCV` values:** EDR writes the skew, the unit serial number and the station number into every trace header. The skew must be the same in all chunks of a station, equal to the skew in the `.raw` file (if there is one), and different from all other stations. The serial number must match the EDR log, the station number the folder name.

**Only a note** (no action needed): differences between the file times and `Date Start` / `Date Stop` in the EDR log `*_seq1.txt`. The log describes the extracted raw chunk, which you extract with some margin.

The notebook also works after step 2, when only the `x`/`y`/`z`/`h` files are left (their trace headers are copies of the EDR ones). You only have to change the cell **Settings**.

## Settings

In [ ]:
from pathlib import Path

DATA_FOLDER = Path("/ur/path/to/data")             # folder that contains one folder per station
STATIONS = ["0989", "1069", "1090", "0498", "0676", "0578"]

## Recording parameters

Fixed for the Trilobit recordings of this course. They are not read from the binary header because EDR writes wrong values there.

In [2]:
import re
import struct
import pandas as pd
from datetime import datetime, timedelta

SAMPLES_PER_TRACE = 30000
SAMPLE_INTERVAL_US = 2000
TRACE_SECONDS = SAMPLES_PER_TRACE * SAMPLE_INTERVAL_US / 1e6     # 60 s
HEADER_BYTES = 3600
TRACE_BYTES = 240 + SAMPLES_PER_TRACE * 4

## Helper functions

In [3]:
def find_chunk_folders(station_folder):
    return sorted(p for p in station_folder.iterdir()
                  if p.is_dir() and re.match(r"^\d+(st|nd|rd|th)Chunk$", p.name))


def find_files(chunk_folder):
    """EDR files (g1/g2/g3/h1) if they exist, otherwise the x/y/z/h files of step 2."""
    stem = f"{chunk_folder.parent.name}_{chunk_folder.name}"
    edr = {c: chunk_folder / f"{stem}_seq1.{c}_segy" for c in ["g1", "g2", "g3", "h1"]}
    if all(p.exists() for p in edr.values()):
        return edr
    converted = {c: chunk_folder / f"{stem}_{c}.segy" for c in "xyzh"}
    if all(p.exists() for p in converted.values()):
        return converted
    return {}


def trace_start_time(trace_header):
    year, day, hour, minute, second = struct.unpack(">5H", trace_header[156:166])
    millisecond, microsecond = struct.unpack(">2H", trace_header[168:172])
    return datetime(year, 1, 1) + timedelta(days=day - 1, hours=hour, minutes=minute, seconds=second,
                                            milliseconds=millisecond, microseconds=microsecond)


def read_file(path):
    """First and last trace time, number of traces, and the Trilobit_RCV values of the first trace."""
    n = (path.stat().st_size - HEADER_BYTES) // TRACE_BYTES
    with open(path, "rb") as f:
        f.seek(HEADER_BYTES)
        first_header = f.read(240)
        f.seek(HEADER_BYTES + (n - 1) * TRACE_BYTES)
        last_header = f.read(240)
    skew_us = struct.unpack(">i", first_header[24:28])[0]       # bytes 25-28
    serial = struct.unpack(">H", first_header[90:92])[0]        # bytes 91-92
    station = struct.unpack(">H", first_header[92:94])[0]       # bytes 93-94
    return trace_start_time(first_header), trace_start_time(last_header), n, skew_us, serial, station


def read_edr_log(chunk_folder):
    """Date Start, Date Stop and unit serial number from *_seq1.txt (None if there is no log)."""
    logs = [p for p in chunk_folder.glob("*_seq1.txt") if not p.name.startswith("._")]
    if not logs:
        return None, None, None
    text = logs[0].read_text(errors="ignore")
    def date(key):
        found = re.search(rf"{key}: \(Day \d+\) \w+ (\w+ +\d+ [\d:]+ \d+)", text)
        return datetime.strptime(found.group(1), "%b %d %H:%M:%S %Y")
    serial = re.search(r"Unit Serial Number: (\d+)", text)
    return date("Date Start"), date("Date Stop"), int(serial.group(1)) if serial else None


def read_raw_skew(chunk_folder):
    """Skew stored in the .raw file at byte 215 (same as get_skew.ps1). None if there is no .raw file."""
    raws = [p for folder in (chunk_folder, chunk_folder.parent) for p in sorted(folder.glob("*.raw"))
            if not p.name.startswith((".", "old_"))]
    if not raws:
        return None
    with open(raws[0], "rb") as f:
        f.seek(214)
        return struct.unpack(">i", f.read(4))[0]

## Read all files

One row per file. Times in UTC, differences in seconds (positive = later than expected).

In [4]:
rows = []
for station in STATIONS:
    chunk_folders = find_chunk_folders(DATA_FOLDER / station)
    if not chunk_folders:
        print(f"{station}: no chunk folders (1stChunk, 2ndChunk, ...) - nothing to check")
    previous_end = None
    for chunk_folder in chunk_folders:
        files = find_files(chunk_folder)
        if not files:
            print(f"{station} {chunk_folder.name}: no g1/g2/g3/h1 or x/y/z/h files - nothing to check")
            continue
        log_start, log_stop, log_serial = read_edr_log(chunk_folder)
        raw_skew = read_raw_skew(chunk_folder)
        infos = {c: read_file(p) for c, p in files.items()}
        times = {c: info[:3] for c, info in infos.items()}
        for component, (first, last, n, skew, serial, station_number) in infos.items():
            end = last + timedelta(seconds=TRACE_SECONDS)
            rows.append(dict(
                station=station, chunk=chunk_folder.name, component=component,
                start=first, end=end, traces=n,
                gap_inside_file=(last - first).total_seconds() - (n - 1) * TRACE_SECONDS,
                gap_to_previous_chunk=None if previous_end is None else (first - previous_end).total_seconds(),
                components_equal=len(set(times.values())) == 1,
                start_vs_log=None if log_start is None else (first - log_start).total_seconds(),
                end_vs_log=None if log_stop is None else (end - log_stop).total_seconds(),
                skew_us=skew, raw_skew_us=raw_skew, serial=serial, serial_log=log_serial, station_number=station_number,
            ))
        previous_end = max(last for _, last, _ in times.values()) + timedelta(seconds=TRACE_SECONDS)

check = pd.DataFrame(rows)
check

0989: no chunk folders (1stChunk, 2ndChunk, ...) - nothing to check


1090: no chunk folders (1stChunk, 2ndChunk, ...) - nothing to check


0676: no chunk folders (1stChunk, 2ndChunk, ...) - nothing to check


,station,chunk,component,start,end,traces,gap_inside_file,gap_to_previous_chunk,components_equal,start_vs_log,end_vs_log,skew_us,raw_skew_us,serial,serial_log,station_number
0,1069,1stChunk,x,2026-07-07 16:00:00,2026-07-17 16:00:00,14400,0.0,NaN,True,0.0,0.0,2385,None,2649,2649,1069
1,1069,1stChunk,y,2026-07-07 16:00:00,2026-07-17 16:00:00,14400,0.0,NaN,True,0.0,0.0,2385,None,2649,2649,1069
2,1069,1stChunk,z,2026-07-07 16:00:00,2026-07-17 16:00:00,14400,0.0,NaN,True,0.0,0.0,2385,None,2649,2649,1069
3,1069,1stChunk,h,2026-07-07 16:00:00,2026-07-17 16:00:00,14400,0.0,NaN,True,0.0,0.0,2385,None,2649,2649,1069
4,1069,2ndChunk,x,2026-07-17 16:00:00,2026-07-27 16:00:00,14400,0.0,0.0,True,0.0,0.0,2385,None,2649,2649,1069
5,1069,2ndChunk,y,2026-07-17 16:00:00,2026-07-27 16:00:00,14400,0.0,0.0,True,0.0,0.0,2385,None,2649,2649,1069
6,1069,2ndChunk,z,2026-07-17 16:00:00,2026-07-27 16:00:00,14400,0.0,0.0,True,0.0,0.0,2385,None,2649,2649,1069
7,1069,2ndChunk,h,2026-07-17 16:00:00,2026-07-27 16:00:00,14400,0.0,0.0,True,0.0,0.0,2385,None,2649,2649,1069
8,1069,3rdChunk,x,2026-07-27 16:00:00,2026-08-03 10:00:00,9720,0.0,0.0,True,0.0,0.0,2385,None,2649,2649,1069
9,1069,3rdChunk,y,2026-07-27 16:00:00,2026-08-03 10:00:00,9720,0.0,0.0,True,0.0,0.0,2385,None,2649,2649,1069


## Find problems

In [5]:
skews_per_station = check.groupby("station")["skew_us"].unique()
stations_per_skew = check[check["skew_us"] != 0].groupby("skew_us")["station"].unique()

problems = []
for _, r in check.iterrows():
    issues = []
    if abs(r["gap_inside_file"]) >= 1e-3:
        issues.append(f"gap inside the file {r['gap_inside_file']:+.3f} s")
    if pd.notna(r["gap_to_previous_chunk"]) and r["gap_to_previous_chunk"] >= 1e-3:
        issues.append(f"gap to the previous chunk {r['gap_to_previous_chunk']:+.3f} s")
    if not r["components_equal"]:
        issues.append("components differ in start, end or number of traces")
    if len(skews_per_station[r["station"]]) > 1:
        issues.append(f"skew {r['skew_us']:+d} us is not the same in all chunks")
    if pd.notna(r["raw_skew_us"]) and r["skew_us"] != r["raw_skew_us"]:
        issues.append(f"skew {r['skew_us']:+d} us, but {int(r['raw_skew_us']):+d} us in the .raw file")
    if r["skew_us"] != 0 and len(stations_per_skew[r["skew_us"]]) > 1:
        others = [s for s in stations_per_skew[r["skew_us"]] if s != r["station"]]
        issues.append(f"skew {r['skew_us']:+d} us is also used by {', '.join(others)}")
    if pd.notna(r["serial_log"]) and r["serial"] != r["serial_log"]:
        issues.append(f"serial {r['serial']} in the trace headers, {int(r['serial_log'])} in the EDR log")
    if r["station_number"] != int(r["station"]):
        issues.append(f"station number {r['station_number']} in the trace headers")
    if issues:
        problems.append(f"  {r['station']} {r['chunk']} {r['component']}: " + ", ".join(issues))

if problems:
    print(f"PROBLEM in {len(problems)} of {len(check)} files:")
    print("\n".join(problems))
else:
    print(f"All {len(check)} files OK: continuous, no gaps, Trilobit_RCV values consistent.")

All 36 files OK: continuous, no gaps, Trilobit_RCV values consistent.


**If there is a problem:** export the affected chunk again in EDR (for a `Trilobit_RCV` problem with the correct skew of that station from `get_skew.ps1`) and run this notebook again. Everything OK: continue with `2_fix_header_and_xyz.ipynb`.

## Notes and Trilobit_RCV values

The notes need no action as long as they match the margins you chose when extracting the raw chunks. Watch only the **start of the first** and the **end of the last** chunk: if they are much later or earlier than planned, data are missing at the beginning or the end.

In [6]:
per_chunk = check.groupby(["station", "chunk"]).first()
for (station, chunk), r in per_chunk.iterrows():
    notes = [f"{label} {r[col]:+.0f} s" for col, label in [("start_vs_log", "start vs. EDR log"), ("end_vs_log", "end vs. EDR log")]
             if pd.notna(r[col]) and abs(r[col]) >= 1e-3]
    if pd.notna(r["gap_to_previous_chunk"]) and r["gap_to_previous_chunk"] <= -1e-3:
        notes.append(f"overlap with the previous chunk {-r['gap_to_previous_chunk']:.0f} s")
    if notes:
        print(f"{station} {chunk}: " + ", ".join(notes))

per_chunk[["start", "end", "traces", "skew_us", "raw_skew_us", "serial", "station_number"]]

start                 end  traces  skew_us  \
station chunk                                                               
0498    1stChunk 2026-07-07 16:00:00 2026-07-17 16:00:00   14400    11461   
        2ndChunk 2026-07-17 16:00:00 2026-07-27 16:00:00   14400    11461   
        3rdChunk 2026-07-27 16:00:00 2026-08-03 10:00:00    9720    11461   
0578    1stChunk 2026-07-07 16:00:00 2026-07-17 16:00:00   14400  -301039   
        2ndChunk 2026-07-17 16:00:00 2026-07-27 16:00:00   14400  -301039   
        3rdChunk 2026-07-27 16:00:00 2026-08-03 10:00:00    9720  -301039   
1069    1stChunk 2026-07-07 16:00:00 2026-07-17 16:00:00   14400     2385   
        2ndChunk 2026-07-17 16:00:00 2026-07-27 16:00:00   14400     2385   
        3rdChunk 2026-07-27 16:00:00 2026-08-03 10:00:00    9720     2385   

                 raw_skew_us  serial  station_number  
station chunk                                         
0498    1stChunk        None    2272             498  
        2ndChunk        None    2272             498  
        3rdChunk        None    2272             498  
0578    1stChunk        None    2141             578  
        2ndChunk        None    2141             578  
        3rdChunk        None    2141             578  
1069    1stChunk        None    2649            1069  
        2ndChunk        None    2649            1069  
        3rdChunk        None    2649            1069